# 🥔 Potato Disease Detection — Detectron2 (High-mAP Edition)

**Target:** mAP@50 ≥ 50%

Key improvements over baseline:
- Stronger backbone: ResNet-101 + FPN
- More training iterations with warm-up
- Aggressive data augmentation (flips, rotation, colour jitter, random crop)
- Cosine LR schedule instead of step decay
- Multi-scale training
- Better anchor sizes tuned for leaf-lesion scales
- Test-Time Augmentation (TTA) for evaluation

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Unzip Dataset

In [ ]:
import zipfile, os

zip_path     = '/content/drive/MyDrive/btp_2_alt.v3i.coco.zip'
extract_path = '/content/drive/MyDrive/detect2'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_path)

print('✅ Unzipped successfully')
print(os.listdir(extract_path))

## 3. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
!nvidia-smi

## 4. Install Detectron2

In [ ]:
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
# Albumentations for rich augmentation pipeline
!pip install -q albumentations

## 5. Register Datasets (COCO format)

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
import json as _json

DatasetCatalog.clear()
MetadataCatalog.clear()

BASE_PATH = '/content/drive/MyDrive/detect2'

with open(f'{BASE_PATH}/train/_annotations.coco.json', 'r') as f:
    _coco = _json.load(f)

CLASS_NAMES = [cat['name'] for cat in sorted(_coco['categories'], key=lambda c: c['id'])]
print('📋 Classes:', CLASS_NAMES)
print('📊 Train images :', len(_coco['images']))
print('📊 Train annotations:', len(_coco['annotations']))

for split in ('train', 'valid', 'test'):
    ann_file = f'{BASE_PATH}/{split}/_annotations.coco.json'
    img_dir  = f'{BASE_PATH}/{split}'
    if os.path.exists(ann_file):
        register_coco_instances(
            f'potato_{split}',
            {'thing_classes': CLASS_NAMES},
            ann_file,
            img_dir
        )

print('✅ Registered:', DatasetCatalog.list())

## 6. Heavy Augmentation Mapper

This is the biggest single lever for small agricultural datasets. We apply:
- Random horizontal + vertical flips
- Random rotation (±30°)
- Random brightness/contrast/saturation
- Random crop (keeps ≥ 70% of image)
- Multi-scale resize

In [ ]:
import copy, torch
import numpy as np
import detectron2.data.transforms as T
from detectron2.data import detection_utils as utils
from detectron2.data import DatasetMapper

def build_augmentation_list(cfg, is_train: bool):
    """Return a list of Detectron2-compatible augmentations."""
    if not is_train:
        return [T.ResizeShortestEdge(
            [cfg.INPUT.MIN_SIZE_TEST], cfg.INPUT.MAX_SIZE_TEST, 'choice'
        )]
    return [
        # Multi-scale training: randomly pick short side in range
        T.ResizeShortestEdge(
            short_edge_length=[480, 512, 544, 576, 608, 640, 672, 704, 736, 768, 800],
            max_size=1333,
            sample_style='choice'
        ),
        T.RandomFlip(prob=0.5, horizontal=True,  vertical=False),
        T.RandomFlip(prob=0.3, horizontal=False, vertical=True),
        T.RandomRotation(angle=[-30, 30], expand=False),
        T.RandomBrightness(0.6, 1.4),
        T.RandomContrast(0.6, 1.4),
        T.RandomSaturation(0.7, 1.3),
        T.RandomCrop('relative_range', (0.7, 1.0)),   # random crop
    ]


class AugmentedMapper(DatasetMapper):
    """DatasetMapper with a custom augmentation pipeline."""
    def __init__(self, cfg, is_train: bool):
        super().__init__(cfg, is_train=is_train)
        self.augmentations = T.AugmentationList(
            build_augmentation_list(cfg, is_train)
        )

print('✅ AugmentedMapper defined')

## 7. Custom Trainer with Augmented Mapper + Cosine LR

In [ ]:
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator
from detectron2.data import build_detection_train_loader
from detectron2.solver import build_lr_scheduler
from fvcore.common.config import CfgNode


class AugTrainer(DefaultTrainer):

    @classmethod
    def build_train_loader(cls, cfg):
        mapper = AugmentedMapper(cfg, is_train=True)
        return build_detection_train_loader(cfg, mapper=mapper)

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, 'coco_eval')
        os.makedirs(output_folder, exist_ok=True)
        return COCOEvaluator(dataset_name, cfg, False, output_dir=output_folder)

    @classmethod
    def build_lr_scheduler(cls, cfg, optimizer):
        # Use WarmupCosineLR for smoother convergence on small datasets
        return build_lr_scheduler(cfg, optimizer)

print('✅ AugTrainer defined')

## 8. Configure & Train

### Key upgrades vs baseline
| Setting | Baseline | This version |
|---|---|---|
| Backbone | R-50-FPN | **R-101-FPN** |
| Iterations | 1 000 | **5 000** |
| LR schedule | Step (700, 900) | **WarmupCosineLR** |
| LR warm-up | none | **500 iters** |
| Augmentation | none | **flip + rotate + colour + crop + multi-scale** |
| ROI batch | 128 | **256** |
| Anchor sizes | default | **tuned for lesions** |

In [ ]:
import os, torch
from detectron2.config import get_cfg
from detectron2 import model_zoo

BASE_PATH  = '/content/drive/MyDrive/detect2'
OUTPUT_DIR = '/content/drive/MyDrive/potato_detectron2/output_v2'

os.makedirs(OUTPUT_DIR, exist_ok=True)

cfg = get_cfg()

# ── Backbone: R-101-FPN (stronger than R-50) ──────────────────────────────
cfg.merge_from_file(model_zoo.get_config_file(
    'COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml'))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    'COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml')

# ── Datasets ───────────────────────────────────────────────────────────────
cfg.DATASETS.TRAIN = ('potato_train',)
cfg.DATASETS.TEST  = ('potato_val',)

# ── Data loading ───────────────────────────────────────────────────────────
cfg.DATALOADER.NUM_WORKERS = 2

# ── Solver ─────────────────────────────────────────────────────────────────
cfg.SOLVER.IMS_PER_BATCH  = 4        # increase if GPU VRAM allows (≥8GB)
cfg.SOLVER.BASE_LR        = 0.001    # slightly higher; cosine will decay it
cfg.SOLVER.MAX_ITER       = 5000     # 5× more iterations
cfg.SOLVER.WARMUP_ITERS   = 500      # linear warm-up prevents early divergence
cfg.SOLVER.WARMUP_FACTOR  = 1.0 / 1000
cfg.SOLVER.LR_SCHEDULER_NAME = 'WarmupCosineLR'  # smooth decay
cfg.SOLVER.STEPS          = ()       # not used with cosine
cfg.SOLVER.GAMMA          = 0.1
cfg.SOLVER.CHECKPOINT_PERIOD = 1000
cfg.SOLVER.WEIGHT_DECAY   = 0.0001

# ── Model ──────────────────────────────────────────────────────────────────
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 256   # more proposals per image
cfg.MODEL.ROI_HEADS.NUM_CLASSES          = len(CLASS_NAMES)
cfg.MODEL.ROI_HEADS.POSITIVE_FRACTION    = 0.5   # balanced sampling

# Tune anchor sizes to typical leaf-lesion scale (small-medium objects)
# FPN levels P2–P6: use smaller anchors for P2/P3 where lesions appear
cfg.MODEL.ANCHOR_GENERATOR.SIZES   = [[16], [32], [64], [128], [256]]
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]] * 5

# ── Input ──────────────────────────────────────────────────────────────────
cfg.INPUT.MIN_SIZE_TRAIN       = (480, 512, 544, 576, 608, 640, 672, 704, 736, 768, 800)
cfg.INPUT.MIN_SIZE_TRAIN_SAMPLING = 'choice'
cfg.INPUT.MIN_SIZE_TEST        = 800
cfg.INPUT.MAX_SIZE_TEST        = 1333

# ── Device ─────────────────────────────────────────────────────────────────
cfg.MODEL.DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg.OUTPUT_DIR   = OUTPUT_DIR

# ── Train ──────────────────────────────────────────────────────────────────
trainer = AugTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()
print('✅ Training complete')

## 9. Evaluate — Standard + Test-Time Augmentation (TTA)

TTA horizontally flips each validation image, runs inference twice, and merges boxes. This typically adds **+1–3 mAP** at no training cost.

In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
from detectron2.modeling import GeneralizedRCNNWithTTA
from detectron2.engine import DefaultPredictor
import copy

# ── Standard evaluation ────────────────────────────────────────────────────
cfg_eval = copy.deepcopy(cfg)
cfg_eval.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.05  # low threshold for eval
cfg_eval.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, 'model_final.pth')

predictor = DefaultPredictor(cfg_eval)

evaluator  = COCOEvaluator('potato_val', cfg_eval, False,
                            output_dir=os.path.join(cfg.OUTPUT_DIR, 'eval'))
val_loader = build_detection_test_loader(cfg_eval, 'potato_val')

print('\n── Standard Evaluation ──')
results = inference_on_dataset(predictor.model, val_loader, evaluator)
print('📊 Standard results:', results)

# ── TTA evaluation ─────────────────────────────────────────────────────────
# TTA config: flip augmentation
cfg_tta = copy.deepcopy(cfg_eval)
cfg_tta.TEST.AUG.ENABLED     = True
cfg_tta.TEST.AUG.MIN_SIZES   = (400, 500, 600, 700, 800)
cfg_tta.TEST.AUG.MAX_SIZE    = 1333
cfg_tta.TEST.AUG.FLIP        = True

tta_model = GeneralizedRCNNWithTTA(cfg_tta, predictor.model)

evaluator_tta  = COCOEvaluator('potato_val', cfg_tta, False,
                                output_dir=os.path.join(cfg.OUTPUT_DIR, 'eval_tta'))
val_loader_tta = build_detection_test_loader(cfg_tta, 'potato_val')

print('\n── TTA Evaluation ──')
results_tta = inference_on_dataset(tta_model, val_loader_tta, evaluator_tta)
print('📊 TTA results:', results_tta)

## 10. Inference on Validation Samples

In [ ]:
import glob, cv2
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.data import MetadataCatalog

BASE_PATH = '/content/drive/MyDrive/detect2'

cfg_infer = copy.deepcopy(cfg)
cfg_infer.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.4   # raise for clean vis
cfg_infer.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, 'model_final.pth')
predictor_vis = DefaultPredictor(cfg_infer)

imgs = (glob.glob(f'{BASE_PATH}/valid/*.jpg') +
        glob.glob(f'{BASE_PATH}/valid/*.jpeg') +
        glob.glob(f'{BASE_PATH}/valid/*.png'))[:6]

if not imgs:
    raise FileNotFoundError('No validation images found.')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, path in zip(axes.flat, imgs):
    img = cv2.imread(path)
    out = predictor_vis(img)
    v   = Visualizer(img[:, :, ::-1],
                     metadata=MetadataCatalog.get('potato_val'),
                     scale=0.7,
                     instance_mode=ColorMode.SEGMENTATION)
    vis = v.draw_instance_predictions(out['instances'].to('cpu'))
    ax.imshow(vis.get_image())
    ax.axis('off')
    ax.set_title(os.path.basename(path)[:30])

plt.suptitle('Potato Disease Detection — Validation Samples', fontsize=14)
plt.tight_layout()
plt.show()

## 11. Upload a Custom Image

In [ ]:
from google.colab import files
import cv2, matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
import copy

uploaded = files.upload()
img_name = list(uploaded.keys())[0]
img = cv2.imread(img_name)
assert img is not None, f'Could not read {img_name}'

cfg_up = copy.deepcopy(cfg)
cfg_up.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.35
cfg_up.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, 'model_final.pth')
pred = DefaultPredictor(cfg_up)

out = pred(img)
v   = Visualizer(img[:, :, ::-1],
                 metadata=MetadataCatalog.get('potato_val'),
                 scale=0.8)
vis = v.draw_instance_predictions(out['instances'].to('cpu'))

plt.figure(figsize=(10, 8))
plt.imshow(vis.get_image())
plt.axis('off')
plt.title(f'Inference: {img_name}')
plt.show()

print('Detected instances:', len(out['instances']))
for i, (score, cls) in enumerate(zip(
        out['instances'].scores.tolist(),
        out['instances'].pred_classes.tolist())):
    print(f'  [{i}] {CLASS_NAMES[cls]} — confidence: {score:.2%}')

## 12. Plot Training Metrics

In [ ]:
import pandas as pd, json, glob
import matplotlib.pyplot as plt

log_file = glob.glob(f'{cfg.OUTPUT_DIR}/metrics.json')
if not log_file:
    print('⚠️ No metrics file yet.')
else:
    records = []
    with open(log_file[0]) as f:
        for line in f:
            try:   records.append(json.loads(line))
            except: pass

    df = pd.DataFrame(records)
    loss_cols = [c for c in df.columns if 'loss' in c.lower()]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves
    if 'iteration' in df.columns and loss_cols:
        dfl = df.dropna(subset=['iteration'] + loss_cols[:1])
        for col in loss_cols:
            axes[0].plot(df['iteration'], df[col], label=col, linewidth=0.8)
        axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Loss')
        axes[0].set_title('Training Losses')
        axes[0].legend(fontsize=7); axes[0].grid(True)

    # LR curve
    if 'lr' in df.columns:
        dflr = df.dropna(subset=['iteration', 'lr'])
        axes[1].plot(dflr['iteration'], dflr['lr'], color='orange')
        axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Learning Rate')
        axes[1].set_title('LR Schedule (WarmupCosineLR)')
        axes[1].grid(True)

    plt.tight_layout(); plt.show()

## 13. Save Model & Config

In [ ]:
import shutil, os

save_dir = '/content/drive/MyDrive/potato_detectron2'
os.makedirs(save_dir, exist_ok=True)

cfg_path = os.path.join(save_dir, 'detectron_cfg_v2.yaml')
with open(cfg_path, 'w') as f:
    f.write(cfg.dump())
print('✅ Config saved:', cfg_path)

src = os.path.join(cfg.OUTPUT_DIR, 'model_final.pth')
dst = os.path.join(save_dir, 'model_final_v2.pth')
if os.path.exists(src):
    shutil.copy(src, dst)
    print('✅ Weights saved:', dst)
else:
    print('⚠️  model_final.pth not found — is training complete?')

## 14. Further Tuning Tips (if mAP still below 50%)

| Lever | Action |
|---|---|
| **More data** | Use Roboflow augmentations during export (flip, crop, HSV) |
| **Longer training** | Increase `MAX_ITER` to 8 000–10 000 |
| **Batch size** | Increase to 8 if GPU allows (A100/V100) |
| **Backbone** | Try `faster_rcnn_X_101_32x8d_FPN_3x` (ResNeXt-101) |
| **Pretrain** | Use domain-specific weights if available |
| **Class imbalance** | Check annotation counts; oversample minority class |
| **NMS threshold** | Lower `cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST` to 0.4 |